# 04 — Bronze to Silver (DQX Profiling + Cleaning + Quarantine)
- Profiles each Bronze table
- Applies data quality rules
- Splits into passed (→ Silver) and failed (→ Quarantine)
- Cleans/standardizes passed records using DataFrame transformations
- Logs DQ results to metadata

In [0]:
# Widget parameters
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_name", "raw_zone", "Schema Name")
dbutils.widgets.text("base_path", "/Volumes/workspace/raw_zone/chinook", "Base Path")
dbutils.widgets.text("table_name", "all", "Table Name")
dbutils.widgets.text("run_id", "", "Run ID (auto-generated if blank)")

catalog = dbutils.widgets.get("catalog_name")
run_id_param = dbutils.widgets.get("run_id")

In [0]:
from pyspark.sql.functions import (
    col, trim, lower, upper, when, lit, current_timestamp,
    regexp_replace, length, to_date, to_timestamp, coalesce,
    count, sum as _sum, isnan, isnull
)
from pyspark.sql.types import IntegerType, DoubleType, StringType, TimestampType, DateType
from datetime import datetime
import uuid

try:
    run_id = dbutils.jobs.taskValues.get(taskKey="task_03_raw_to_bronze", key="run_id")
except:
    run_id = run_id_param if run_id_param else str(uuid.uuid4())[:8]

print(f"Run ID: {run_id}")

Run ID: b14d4bda


## 1. DQ Rules Definition (Metadata-Driven)

In [0]:
# DQ rules per table: list of (rule_name, description, filter_expr_for_failures)
# Records matching the filter expression are FAILURES and go to quarantine
dq_rules = {
    "customer": [
        ("not_null_customerid", "CustomerId must not be null", "CustomerId IS NULL"),
        ("not_null_email", "Email must not be null", "Email IS NULL"),
        ("not_null_firstname", "FirstName must not be null", "FirstName IS NULL"),
        ("not_null_lastname", "LastName must not be null", "LastName IS NULL"),
        ("valid_email_format", "Email must contain @", "Email IS NOT NULL AND Email NOT LIKE '%@%'"),
    ],
    "invoice": [
        ("not_null_invoiceid", "InvoiceId must not be null", "InvoiceId IS NULL"),
        ("not_null_customerid", "CustomerId must not be null", "CustomerId IS NULL"),
        ("valid_total", "Total must be >= 0", "Total < 0"),
        ("not_null_invoicedate", "InvoiceDate must not be null", "InvoiceDate IS NULL"),
    ],
    "invoiceline": [
        ("not_null_invoicelineid", "InvoiceLineId must not be null", "InvoiceLineId IS NULL"),
        ("not_null_invoiceid", "InvoiceId must not be null", "InvoiceId IS NULL"),
        ("not_null_trackid", "TrackId must not be null", "TrackId IS NULL"),
        ("valid_quantity", "Quantity must be > 0", "Quantity <= 0"),
        ("valid_unitprice", "UnitPrice must be >= 0", "UnitPrice < 0"),
    ],
    "track": [
        ("not_null_trackid", "TrackId must not be null", "TrackId IS NULL"),
        ("not_null_name", "Name must not be null", "Name IS NULL"),
        ("valid_unitprice", "UnitPrice must be >= 0", "UnitPrice < 0"),
        ("valid_milliseconds", "Milliseconds must be > 0", "Milliseconds <= 0"),
    ],
    "album": [
        ("not_null_albumid", "AlbumId must not be null", "AlbumId IS NULL"),
        ("not_null_title", "Title must not be null", "Title IS NULL"),
    ],
    "artist": [
        ("not_null_artistid", "ArtistId must not be null", "ArtistId IS NULL"),
    ],
    "employee": [
        ("not_null_employeeid", "EmployeeId must not be null", "EmployeeId IS NULL"),
        ("not_null_lastname", "LastName must not be null", "LastName IS NULL"),
    ],
    "genre": [
        ("not_null_genreid", "GenreId must not be null", "GenreId IS NULL"),
    ],
    "mediatype": [
        ("not_null_mediatypeid", "MediaTypeId must not be null", "MediaTypeId IS NULL"),
    ],
    "playlist": [
        ("not_null_playlistid", "PlaylistId must not be null", "PlaylistId IS NULL"),
    ],
    "playlisttrack": [
        ("not_null_playlistid", "PlaylistId must not be null", "PlaylistId IS NULL"),
        ("not_null_trackid", "TrackId must not be null", "TrackId IS NULL"),
    ],
}

## 2. Profiling Helper

In [0]:
def profile_table(df, table_name):
    """Run profiling summary: null counts, distinct counts, data types."""
    print(f"\n--- Profiling: {table_name} ---")
    total = df.count()
    print(f"  Total rows: {total}")
    print(f"  Columns: {len(df.columns)}")

    for c in df.columns:
        null_count = df.filter(col(c).isNull()).count()
        distinct_count = df.select(c).distinct().count()
        if null_count > 0:
            print(f"  ⚠ {c}: {null_count} nulls ({round(null_count/max(total,1)*100,1)}%)")

    return total

## 3. Cleaning Functions (DataFrame Transformations)

In [0]:
def clean_string_columns(df):
    """Trim whitespace and normalize empty strings to null for all string columns."""
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                when(
                    trim(col(field.name)) == "", None
                ).otherwise(
                    trim(col(field.name))
                )
            )
    return df

def standardize_country(df):
    """Standardize country names if Country column exists."""
    if "Country" in df.columns:
        df = df.withColumn("Country", trim(col("Country")))
    return df

def standardize_email(df):
    """Lowercase email if Email column exists."""
    if "Email" in df.columns:
        df = df.withColumn("Email", lower(trim(col("Email"))))
    return df

def remove_exact_duplicates(df):
    """Remove exact duplicate rows."""
    before = df.count()
    df = df.dropDuplicates()
    after = df.count()
    if before != after:
        print(f"    Removed {before - after} exact duplicates")
    return df

def apply_silver_cleaning(df, table_name):
    """Apply all cleaning transformations for Silver layer."""
    # 1. Trim all strings and normalize empty → null
    df = clean_string_columns(df)

    # 2. Table-specific standardization
    if table_name == "customer":
        df = standardize_country(df)
        df = standardize_email(df)

    if table_name == "employee":
        df = standardize_email(df)

    # 3. Remove exact duplicates
    df = remove_exact_duplicates(df)

    return df

## 4. Process Each Bronze Table

In [0]:
df_metadata = spark.sql(f"""
    SELECT table_name, primary_key_col
    FROM {catalog}.metadata.pipeline_parent_metadata
    WHERE active_flag = 'Y'
    ORDER BY load_sequence
""")

tables = df_metadata.collect()
silver_results = []

In [0]:
for row in tables:
    start_time = datetime.now()
    table_name = row.table_name
    pk_col = row.primary_key_col

    try:
        # Read Bronze
        bronze_table = f"{catalog}.bronze.{table_name}"
        df_bronze = spark.read.table(bronze_table)
        total_count = df_bronze.count()

        print(f"\n{'='*50}")
        print(f"Processing: {table_name} ({total_count} rows)")
        print(f"{'='*50}")

        # --- PROFILING ---
        profile_table(df_bronze, table_name)

        # --- DQ VALIDATION ---
        rules = dq_rules.get(table_name, [])
        failed_df = None
        total_failed = 0

        for rule_name, rule_desc, fail_expr in rules:
            # Count failures for this rule
            rule_failures = df_bronze.filter(fail_expr).count()

            # Log each rule to DQ execution log
            spark.sql(f"""
                INSERT INTO {catalog}.metadata.dq_execution_log
                VALUES (
                    '{run_id}', '{table_name}', '{rule_name}', '{rule_desc}',
                    {total_count}, {total_count - rule_failures}, {rule_failures},
                    current_timestamp()
                )
            """)

            if rule_failures > 0:
                print(f"  DQ FAIL: {rule_name} — {rule_failures} failures")
                rule_failed = df_bronze.filter(fail_expr)
                if failed_df is None:
                    failed_df = rule_failed
                else:
                    failed_df = failed_df.union(rule_failed).distinct()

        # Split passed / failed
        if failed_df is not None:
            total_failed = failed_df.count()
            df_passed = df_bronze.subtract(failed_df)
        else:
            total_failed = 0
            df_passed = df_bronze

        passed_count = df_passed.count()
        print(f"\n  DQ Summary: total={total_count}, passed={passed_count}, failed={total_failed}")

        # --- QUARANTINE failed records ---
        if total_failed > 0:
            quarantine_table = f"{catalog}.quarantine.{table_name}"
            failed_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(quarantine_table)
            print(f"  Quarantined {total_failed} records → {quarantine_table}")

        # --- SILVER CLEANING on passed records only ---
        print(f"  Applying Silver cleaning transformations...")
        df_clean = apply_silver_cleaning(df_passed, table_name)
        silver_count = df_clean.count()

        # Write to Silver
        silver_table = f"{catalog}.silver.{table_name}"
        df_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table)
        print(f"  ✓ Silver written: {silver_count} rows → {silver_table}")

        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        silver_results.append({
            "table_name": table_name,
            "bronze_count": total_count,
            "passed_dq": passed_count,
            "failed_dq": total_failed,
            "silver_count": silver_count,
            "status": "success",
            "duration": duration
        })

        # Log metrics
        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'bronze_to_silver',
                current_timestamp(), 'success',
                {total_count}, {silver_count},
                {passed_count}, {total_failed},
                '{silver_table}', NULL,
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)

    except Exception as e:
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()
        error_msg = str(e).replace("'", "''")[:500]
        print(f"  ✗ {table_name}: FAILED — {str(e)[:200]}")

        silver_results.append({
            "table_name": table_name,
            "bronze_count": 0,
            "passed_dq": 0,
            "failed_dq": 0,
            "silver_count": 0,
            "status": "failed",
            "duration": duration
        })

        spark.sql(f"""
            INSERT INTO {catalog}.metadata.pipeline_child_metrics
            VALUES (
                '{run_id}', '{table_name}', 'bronze_to_silver',
                current_timestamp(), 'failed',
                0, 0, 0, 0, '', '{error_msg}',
                '{start_time}', '{end_time}', {duration},
                current_timestamp()
            )
        """)


Processing: album (347 rows)

--- Profiling: album ---
  Total rows: 347
  Columns: 3

  DQ Summary: total=347, passed=347, failed=0
  Applying Silver cleaning transformations...
  ✓ Silver written: 347 rows → workspace.silver.album

Processing: artist (275 rows)

--- Profiling: artist ---
  Total rows: 275
  Columns: 2

  DQ Summary: total=275, passed=275, failed=0
  Applying Silver cleaning transformations...
  ✓ Silver written: 275 rows → workspace.silver.artist

Processing: customer (59 rows)

--- Profiling: customer ---
  Total rows: 59
  Columns: 13
  ⚠ Company: 49 nulls (83.1%)
  ⚠ State: 29 nulls (49.2%)
  ⚠ PostalCode: 4 nulls (6.8%)
  ⚠ Phone: 1 nulls (1.7%)
  ⚠ Fax: 47 nulls (79.7%)

  DQ Summary: total=59, passed=59, failed=0
  Applying Silver cleaning transformations...
  ✓ Silver written: 59 rows → workspace.silver.customer

Processing: employee (8 rows)

--- Profiling: employee ---
  Total rows: 8
  Columns: 15
  ⚠ ReportsTo: 1 nulls (12.5%)

  DQ Summary: total=8, pass

## 5. Silver Summary

In [0]:
import pandas as pd

summary_df = pd.DataFrame(silver_results)
print(f"\n{'='*60}")
print(f"BRONZE → SILVER SUMMARY — Run ID: {run_id}")
print(f"{'='*60}")
print(f"Total tables:      {len(silver_results)}")
print(f"Successful:        {sum(1 for r in silver_results if r['status'] == 'success')}")
print(f"Total quarantined: {sum(r['failed_dq'] for r in silver_results)}")
print(f"{'='*60}")

display(spark.createDataFrame(summary_df))


BRONZE → SILVER SUMMARY — Run ID: b14d4bda
Total tables:      11
Successful:        11
Total quarantined: 0


table_name,bronze_count,passed_dq,failed_dq,silver_count,status,duration
album,347,347,0,347,success,13.423333
artist,275,275,0,275,success,10.267729
customer,59,59,0,59,success,25.542992
employee,8,8,0,8,success,20.752109
genre,25,25,0,25,success,9.253603
invoice,412,412,0,412,success,19.001359
invoiceline,2240,2240,0,2240,success,17.888294
mediatype,5,5,0,5,success,9.002617
playlist,18,18,0,18,success,9.26948
playlisttrack,8715,8715,0,8715,success,10.734259


## 6. View DQ Log

In [0]:
display(spark.sql(f"""
    SELECT table_name, rule_name, rule_description, total_processed, passed_count, failed_count
    FROM {catalog}.metadata.dq_execution_log
    WHERE run_id = '{run_id}'
    ORDER BY table_name, rule_name
"""))

table_name,rule_name,rule_description,total_processed,passed_count,failed_count
album,not_null_albumid,AlbumId must not be null,347,347,0
album,not_null_title,Title must not be null,347,347,0
artist,not_null_artistid,ArtistId must not be null,275,275,0
customer,not_null_customerid,CustomerId must not be null,59,59,0
customer,not_null_email,Email must not be null,59,59,0
customer,not_null_firstname,FirstName must not be null,59,59,0
customer,not_null_lastname,LastName must not be null,59,59,0
customer,valid_email_format,Email must contain @,59,59,0
employee,not_null_employeeid,EmployeeId must not be null,8,8,0
employee,not_null_lastname,LastName must not be null,8,8,0


## 7. View Quarantine (if any)

In [0]:
# List any quarantine tables that were created
quarantine_tables = [r.table_name for r in silver_results if r["failed_dq"] > 0]
for qt in quarantine_tables:
    print(f"\nQuarantine: {catalog}.quarantine.{qt}")
    display(spark.sql(f"SELECT * FROM {catalog}.quarantine.{qt} LIMIT 5"))

In [0]:
dbutils.jobs.taskValues.set(key="run_id", value=run_id)